In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
df = spark.read.parquet(f"{presentation_folder_path}/race_results").filter("race_year = 2019")

In [0]:
from pyspark.sql.functions import count, countDistinct, sum

In [0]:
t()

In [0]:
df.select(count("*")).show()

In [0]:
display(df.select(countDistinct("race_name")))


In [0]:
df.filter("driver_name ='Lewis Hamilton'").select(sum("points"), countDistinct("race_name")) \
.withColumnRenamed("sum(points)", "total_points") \
.withColumnRenamed("count(DISTINCT race_name)", "number_of_races") \
.show()

In [0]:
df \
  .groupBy("driver_name") \
  .agg(sum("points"), countDistinct("race_name")) \
  .withColumnRenamed("sum(points)", "total_points") \
  .withColumnRenamed("count(DISTINCT race_name)","number_of_races") \
  .show()

## Window functions

In [0]:
df = spark.read.parquet(f"{presentation_folder_path}/race_results").filter("race_year in (2019, 2020)")
display(df)

In [0]:
from pyspark.sql.window import Window
df = df \
  .groupBy("race_year", "driver_name") \
  .agg(sum("points"), countDistinct("race_name")) \
  .withColumnRenamed("sum(points)", "total_points") \
  .withColumnRenamed("count(DISTINCT race_name)","number_of_races") \
  .orderBy("total_points", ascending=False)

df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, desc, rank
driverRankSpec = Window.partitionBy("race_year").orderBy(desc(col("total_points")))
df \
  .withColumn("rank", rank().over(driverRankSpec)) \
  .show(100)